# AoC 2024 Day 5 — Print Queue

**Spark — self-join for pairwise constraints; sorting by counting**

Puzzle: <https://adventofcode.com/2024/day/5>

---

> **On puzzle text and inputs.** Advent of Code is Eric Wastl's work, and he asks that puzzle text and per-user inputs not be redistributed. So this notebook carries a summary in my own words plus the *published* example, and pulls the real input at runtime from a local cache that is gitignored. Read the puzzle at the link above.

---

## The puzzle

Two blocks: ordering rules `X|Y` meaning page X must be printed before page Y, then a list of updates (each a comma-separated page list).

- **Part 1** — find the updates already in a valid order, and sum their **middle** page numbers.
- **Part 2** — take the *incorrectly* ordered updates, reorder each one correctly, and sum their middle pages.

## The approach

Two ideas carry this one.

**Part 1 — a violation is a join.** Explode each update to `(update, position, page)`, self-join it on `pos_a < pos_b` to get every ordered pair within an update, then join *that* to the rules **reversed**. Any surviving row is a broken rule, so an update is correct exactly when it produces no rows — which is a `left_anti` join.

**Part 2 — sorting without a sort.** The single-node instinct is a comparator (`cmp_to_key`), and that is what a plain-Python solution does. But the correct index of a page is determined by *counting*: if page *p* must precede *k* of the other pages in its update, *p* lands at index *n−1−k*. So the middle page — index *n//2* — is the page that precedes exactly *n//2* others.

That is a **groupBy, not a sort**, and it never materialises the ordering at all. Since the puzzle only ever asks for the middle element, computing the full order would be wasted work.

## Setup

Connect to the cluster's Spark Connect endpoint and import the solution.

In [ ]:
import sys

sys.path.insert(0, '..')  # so `aoc_spark` resolves when running from notebooks/

from aoc_spark.session import get_spark
from aoc_spark.inputs import get_input
from aoc_spark.y2024 import day05

spark = get_spark('aoc-2024-day05')
print('Spark', spark.version)

## The published example

The same data the test suite asserts on.

In [ ]:
EXAMPLE = """\
47|53
97|13
97|61
97|47
75|29
61|13
75|53
29|13
97|29
53|29
61|53
97|53
61|29
47|13
75|47
97|75
47|61
75|61
47|29
75|13
53|13

75,47,61,53,29
97,61,53,29,13
75,29,13
75,97,47,61,53
61,13,29
97,13,75,29,47
"""

print('part 1:', day05.part1(spark, EXAMPLE), '(expected 143)')
print('part 2:', day05.part2(spark, EXAMPLE), '(expected 123)')

### Violations, as rows

Each row below is a concrete broken promise: `page_a` sits before `page_b`, but a rule says `page_b` must come first.

In [ ]:
from pyspark.sql import functions as F

rules, pages = day05.parse(spark, EXAMPLE)
rules.show(5)
pages.orderBy('update_id', 'pos').show(10)

a = pages.select('update_id', F.col('pos').alias('pos_a'), F.col('page').alias('page_a'))
b = pages.select(
    F.col('update_id').alias('uid_b'),
    F.col('pos').alias('pos_b'),
    F.col('page').alias('page_b'),
)
pairs = a.join(b, (F.col('update_id') == F.col('uid_b')) & (F.col('pos_a') < F.col('pos_b')))
pairs.join(
    rules, (F.col('before') == F.col('page_b')) & (F.col('after') == F.col('page_a'))
).select('update_id', 'page_a', 'page_b', 'before', 'after').show()

## The real input

`get_input` is cache-first: local gitignored file → Postgres → adventofcode.com. In practice it hits the local file and never touches the network.

In [ ]:
import time

data = get_input(2024, 5)
print(f'input: {len(data):,} chars, {len(data.splitlines()):,} lines')

for part in (1, 2):
    fn = getattr(day05, f'part{part}')
    started = time.perf_counter()
    answer = fn(spark, data)
    print(f'part {part}: {answer}  ({(time.perf_counter() - started) * 1000:.0f} ms)')

## Notes & gotchas

- The counting argument in part 2 relies on the rules being **total** within each update — every pair of pages in an update is covered by some rule. AoC's input satisfies this; a general topological sort would not be able to assume it. Worth checking that assumption on your own input before trusting the shortcut.
- `left_anti` is the join to reach for whenever the question is "rows *without* a match" — it avoids the `distinct()` + `NOT IN` shape that would otherwise appear.
- Middle index is `size // 2` with integer division; every AoC update has an odd length, so there is always an unambiguous middle.